# Character-Level LSTM Text Generation

Trains a char-level LSTM on tiny-shakespeare using RecPulse's `LSTMCell` — a Module composed
entirely from existing autograd ops (matmul, broadcast add, chunk, sigmoid, tanh). Truncated
BPTT unrolls 40 timesteps per batch; the autograd graph for one step spans ~2,200 nodes.

Training runs on Athena (A100) via `char_lstm_athena.sh`, which wraps `char_lstm_train.py`.
This notebook documents the example and the results of job **2853457** (2026-08-02).

| | |
|---|---|
| Corpus | tiny-shakespeare, 1,115,394 chars, vocab 65 |
| Model | Embedding(65, 64) → LSTMCell(64, 256) → Linear(256, 65) — 350,593 params |
| Training | 4,000 iters, batch 64, seq 40, Adam lr 2e-3 |
| Wall clock | 29.6 min on one A100 |
| Loss | train 3.23 → **1.41**, val **1.64** |

## Model

The full training script is `char_lstm_train.py`; the model is just:

In [ ]:
import sys
sys.path.insert(0, '..')
import recpulse_cuda as rp
from recpulse.module import Module, LSTMCell, Linear, Embedding


class CharLSTM(Module):
    def __init__(self, vocab, embed_dim, hidden):
        super().__init__()
        self.embed = Embedding(vocab, embed_dim)
        self.cell = LSTMCell(embed_dim, hidden)
        self.head = Linear(hidden, vocab)

    def forward(self, idx_steps):
        state = None
        logits = []
        for idx in idx_steps:
            x = self.embed(idx)
            h, c = self.cell(x, state)
            state = (h, c)
            logits.append(self.head(h))
        return logits

The per-sequence loss averages `op_cross_entropy_loss(..., 'mean', 1)` over the 40 unrolled
timesteps, and one `backward()` propagates through the whole unrolled graph.

## Training curve (job 2853457)

```
iter   100 | loss 3.2269      --- val @  500: 2.0261
iter   500 | loss 2.0243      --- val @ 1000: 1.8283
iter  1000 | loss 1.7185      --- val @ 1500: 1.7611
iter  1500 | loss 1.5963      --- val @ 2000: 1.6807
iter  2000 | loss 1.5300      --- val @ 2500: 1.6414
iter  3000 | loss 1.4600      --- val @ 3000: 1.7099
iter  3500 | loss 1.4210      --- val @ 3500: 1.6764
iter  4000 | loss 1.4068      --- val @ 4000: 1.6368
```

## Generated sample (iter 4000, temperature 0.8)

```
There, as it over of the unkings, shall say you have no more the house of ever.

HENRY BOLINGBROKE:
You are you, for less not too fear:
And the new cross and succerate gives her spoke.

BAPTISTA:
What, she was a princes, hand me how be to-day,
Smeliant I see this mind the rest,
The bed, for I speak
```

The model has learned the play structure (speaker names in caps followed by a colon, then
verse lines), real character names, and largely valid English morphology — the expected
quality for a single-layer char-LSTM at this loss.

## Reproduce

```bash
# on Athena, from cuda_utils/ (corpus must exist at data/tinyshakespeare.txt)
sbatch examples/char_lstm_athena.sh
```

Checkpoint (`char_lstm.rpt`), per-iteration losses, and samples land in `data/char_lstm_run/`.